# Publication Delays Management

This notebook demonstrates the usage of the `tsforecast.delays` module for managing publication delays in economic time series data.

## Objectives

1. **Data Comparison**: Compare new datasets with existing data to identify new observations
2. **Delay Calculation**: Calculate publication delays for economic indicators
3. **Delay Application**: Apply delays using sklearn-compatible transformers
4. **Mixed Frequencies**: Handle both monthly and quarterly data

## Key Features

- **ReleaseDataManager**: Compare datasets and calculate delays
- **ReleaseDelayCalculator**: Analyze delay statistics from historical data
- **ReleaseDelayTransformer**: Apply delays with 'shift' or 'mask' modes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import delays module components
from tsforecast.delays import (
    ReleaseDataManager,
    ReleaseDelayCalculator, 
    ReleaseDelayTransformer,
    create_delay_transformer_from_dict
)

## 1. Generating Synthetic Economic Data

Let's create fake mixed frequency economic data to demonstrate the functionality:

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Create monthly dates from 2020 to 2024
monthly_dates = pd.date_range('2020-01-01', '2024-01-01', freq='MS')
quarterly_dates = pd.date_range('2020-01-01', '2024-01-01', freq='QS')

def generate_economic_series(dates, base_value=100, trend=0.02, volatility=0.05):
    """Generate realistic economic time series"""
    n = len(dates)
    # Trend component
    trend_component = np.cumsum(np.random.normal(trend/12, volatility/4, n))
    # Cyclical component
    cycle = 0.1 * np.sin(2 * np.pi * np.arange(n) / 12) 
    # Random noise
    noise = np.random.normal(0, volatility, n)
    
    return base_value * np.exp(trend_component + cycle + noise)

# Generate monthly indicators
monthly_data = pd.DataFrame({
    'date': monthly_dates,
    'country': 'US',
    'inflation_rate': generate_economic_series(monthly_dates, 2.0, 0.001, 0.02),
    'unemployment_rate': generate_economic_series(monthly_dates, 5.0, -0.001, 0.03),
    'industrial_production': generate_economic_series(monthly_dates, 100, 0.002, 0.04)
})

# Generate quarterly indicators
quarterly_data = pd.DataFrame({
    'date': quarterly_dates,
    'country': 'US', 
    'gdp_growth': generate_economic_series(quarterly_dates, 2.5, 0.0005, 0.015),
    'government_debt': generate_economic_series(quarterly_dates, 80, 0.01, 0.02)
})

print("Monthly data shape:", monthly_data.shape)
print("Quarterly data shape:", quarterly_data.shape)
print("\nMonthly data sample:")
print(monthly_data.head())
print("\nQuarterly data sample:")
print(quarterly_data.head())

## 2. Data Comparison and Delay Calculation

Let's simulate how new data releases work and calculate publication delays:

In [ ]:
# Create Release Data Manager
manager = ReleaseDataManager(
    time_col='date',
    panel_cols=['country'],
    default_reference_point='end'
)

# Simulate "existing" data (data up to mid-2023)
existing_monthly = monthly_data[monthly_data['date'] <= '2023-06-01'].copy()
existing_quarterly = quarterly_data[quarterly_data['date'] <= '2023-04-01'].copy()

# Simulate "new" data with some additional observations
new_monthly = monthly_data[monthly_data['date'] <= '2023-08-01'].copy()
new_quarterly = quarterly_data[quarterly_data['date'] <= '2023-07-01'].copy()

print("Existing monthly data ends:", existing_monthly['date'].max())
print("New monthly data ends:", new_monthly['date'].max())
print("Existing quarterly data ends:", existing_quarterly['date'].max())
print("New quarterly data ends:", new_quarterly['date'].max())

In [ ]:
# Compare datasets and calculate delays for monthly data
download_date_monthly = datetime(2023, 9, 15)  # Downloaded on Sept 15
monthly_comparison = manager.compare_and_calculate_delays(
    new_data=new_monthly,
    existing_data=existing_monthly,
    download_date=download_date_monthly,
    reference_point='end'
)

print("Monthly comparison results:")
print(f"New observations found: {monthly_comparison['new_observations_count']}")
print(f"Delays calculated: {monthly_comparison['delays_calculated']}")
print("\nFirst few delay records:")
for record in monthly_comparison['delay_records'][:3]:
    print(f"- {record['indicator_name']}: {record['release_delay_days']:.1f} days delay")

In [ ]:
# Compare datasets for quarterly data
download_date_quarterly = datetime(2023, 10, 20)  # Downloaded on Oct 20
quarterly_comparison = manager.compare_and_calculate_delays(
    new_data=new_quarterly,
    existing_data=existing_quarterly,
    download_date=download_date_quarterly,
    reference_point='end'
)

print("Quarterly comparison results:")
print(f"New observations found: {quarterly_comparison['new_observations_count']}")
print(f"Delays calculated: {quarterly_comparison['delays_calculated']}")
print("\nFirst few delay records:")
for record in quarterly_comparison['delay_records'][:3]:
    print(f"- {record['indicator_name']}: {record['release_delay_days']:.1f} days delay")

## 3. Delay Statistics Analysis

Let's analyze the delay patterns using the ReleaseDelayCalculator:

In [ ]:
# Combine all delay records
all_delay_records = monthly_comparison['delay_records'] + quarterly_comparison['delay_records']

# Create DataFrame for delay analysis
delay_df = pd.DataFrame(all_delay_records)

# Create delay calculator
calculator = ReleaseDelayCalculator(
    delay_data=delay_df,
    default_reference_point='end',
    min_observations=1  # Low threshold for demo
)

# Calculate median delays by indicator
median_delays = calculator.calculate_median_delays(group_by_entity=False)

print("Median delays by indicator:")
for indicator, delay in median_delays.items():
    print(f"- {indicator}: {delay:.1f} days")

In [ ]:
# Calculate comprehensive statistics
comprehensive_stats = calculator.calculate_comprehensive_stats(group_by_entity=False)

print("Comprehensive delay statistics:")
for indicator, stats in comprehensive_stats.items():
    print(f"\n{indicator}:")
    print(f"  Median: {stats['median']:.1f} days")
    print(f"  Mean: {stats['mean']:.1f} days")
    print(f"  Std: {stats['std']:.1f} days")
    print(f"  Count: {stats['count']} observations")

## 4. Visualization of Delays

Let's visualize the delay patterns:

In [ ]:
# Create visualization of delays by indicator
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Median delays by indicator
indicators = list(median_delays.keys())
delays = list(median_delays.values())

axes[0].bar(indicators, delays, color=['skyblue', 'lightcoral', 'lightgreen', 'orange', 'pink'])
axes[0].set_title('Median Publication Delays by Indicator')
axes[0].set_ylabel('Delay (days)')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Distribution of delays
if not delay_df.empty:
    delay_df['release_delay_days'].hist(bins=15, alpha=0.7, ax=axes[1])
    axes[1].set_title('Distribution of Publication Delays')
    axes[1].set_xlabel('Delay (days)')
    axes[1].set_ylabel('Frequency')
    axes[1].axvline(delay_df['release_delay_days'].median(), color='red', 
                   linestyle='--', label=f'Median: {delay_df["release_delay_days"].median():.1f} days')
    axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Applying Delays with Transformers

Now let's demonstrate how to apply delays using the sklearn-compatible transformers:

In [ ]:
# Create a combined dataset for transformation
# For simplicity, let's resample quarterly data to monthly frequency
quarterly_monthly = quarterly_data.set_index('date').resample('MS').ffill().reset_index()
quarterly_monthly['country'] = 'US'

# Merge monthly and quarterly data
combined_data = pd.merge(monthly_data, quarterly_monthly, on=['date', 'country'], how='left')
combined_data = combined_data.fillna(method='ffill')  # Forward fill quarterly data

print("Combined data shape:", combined_data.shape)
print("\nCombined data sample:")
print(combined_data.head())

In [ ]:
# Define realistic publication delays for our indicators
publication_delays = {
    'inflation_rate': 15.0,        # CPI usually released ~15 days after month end
    'unemployment_rate': 5.0,      # Employment data released ~5 days after month end  
    'industrial_production': 20.0, # Industrial production ~20 days
    'gdp_growth': 45.0,            # GDP released ~45 days after quarter end
    'government_debt': 60.0        # Government finance stats ~60 days
}

print("Publication delays (days):")
for indicator, delay in publication_delays.items():
    print(f"- {indicator}: {delay} days")

### 5.1 Shift Mode Transformation

In shift mode, data is shifted forward in time according to publication delays:

In [ ]:
# Create transformer with shift mode
shift_transformer = create_delay_transformer_from_dict(
    delays_dict=publication_delays,
    mode='shift',
    prediction_date='2023-12-01',
    time_col='date',
    panel_cols=['country']
)

# Prepare data for transformation (focus on 2023 data)
transform_data = combined_data[combined_data['date'] >= '2023-01-01'].copy()

# Apply shift transformation
shifted_data = shift_transformer.fit_transform(transform_data)

print("Original data shape:", transform_data.shape)
print("Shifted data shape:", shifted_data.shape)
print("\nTransformation summary:")
summary = shift_transformer.get_transformation_summary()
for key, value in summary.items():
    if key != 'delays_applied':  # Skip the detailed delays dict for readability
        print(f"- {key}: {value}")

### 5.2 Mask Mode Transformation

In mask mode, data that wouldn't be available at prediction time is masked with NaN:

In [ ]:
# Create transformer with mask mode
mask_transformer = create_delay_transformer_from_dict(
    delays_dict=publication_delays,
    mode='mask',
    prediction_date='2023-12-01',  # Prediction made on Dec 1, 2023
    time_col='date',
    panel_cols=['country']
)

# Apply mask transformation
masked_data = mask_transformer.fit_transform(transform_data)

print("Mask transformation results:")
print(f"Original data points: {transform_data.iloc[:, 2:].notna().sum().sum()}")
print(f"Masked data points: {masked_data.iloc[:, 2:].notna().sum().sum()}")
print(f"Percentage of data available: {masked_data.iloc[:, 2:].notna().sum().sum() / transform_data.iloc[:, 2:].notna().sum().sum() * 100:.1f}%")

### 5.3 Comparison of Available Data

Let's compare what data would be available for different indicators on our prediction date:

In [ ]:
# Compare last available data point for each indicator
prediction_date = pd.to_datetime('2023-12-01')

print(f"Data availability as of {prediction_date.strftime('%Y-%m-%d')}:\n")

indicators = ['inflation_rate', 'unemployment_rate', 'industrial_production', 'gdp_growth', 'government_debt']

for indicator in indicators:
    # Original last available
    original_last = transform_data[transform_data[indicator].notna()]['date'].max()
    
    # Masked last available
    masked_last = masked_data[masked_data[indicator].notna()]['date'].max()
    
    delay = publication_delays[indicator]
    
    print(f"{indicator}:")
    print(f"  Delay: {delay} days")
    print(f"  Original last: {original_last.strftime('%Y-%m-%d')}")
    print(f"  Available last: {masked_last.strftime('%Y-%m-%d')}")
    print(f"  Data lag: {(prediction_date - masked_last).days} days")
    print()

## 6. Practical Example: Economic Nowcasting

Let's demonstrate a practical nowcasting scenario where we need to predict current quarter GDP using available indicators with realistic delays:

In [ ]:
# Scenario: It's December 1, 2023, and we want to nowcast Q4 2023 GDP
# Different indicators have different delays

nowcast_date = pd.to_datetime('2023-12-01')
target_quarter = 'Q4 2023'

print(f"Nowcasting Scenario: {nowcast_date.strftime('%Y-%m-%d')}")
print(f"Target: {target_quarter} GDP Growth")
print("\nIndicator availability with realistic delays:")

# Create different transformers for different scenarios
scenarios = {
    'Ideal (no delays)': {},  # No delays
    'Realistic delays': publication_delays,
    'Pessimistic delays': {k: v * 1.5 for k, v in publication_delays.items()}  # 50% longer delays
}

availability_summary = {}

for scenario_name, delays in scenarios.items():
    if delays:  # If there are delays
        transformer = create_delay_transformer_from_dict(
            delays_dict=delays,
            mode='mask',
            prediction_date=nowcast_date,
            time_col='date',
            panel_cols=['country']
        )
        scenario_data = transformer.fit_transform(transform_data)
    else:  # No delays scenario
        scenario_data = transform_data.copy()
    
    # Calculate data availability for Q4 2023 (Oct-Dec)
    q4_data = scenario_data[
        (scenario_data['date'] >= '2023-10-01') & 
        (scenario_data['date'] < '2024-01-01')
    ]
    
    availability = {}
    for indicator in indicators:
        available_months = q4_data[indicator].notna().sum()
        availability[indicator] = available_months / 3 * 100  # Percentage of Q4 available
    
    availability_summary[scenario_name] = availability

# Display results
import pandas as pd
availability_df = pd.DataFrame(availability_summary).round(1)
print("\nData Availability for Q4 2023 (% of quarter):")
print(availability_df)

In [ ]:
# Visualize the impact of delays on data availability
fig, ax = plt.subplots(figsize=(12, 8))

availability_df.plot(kind='bar', ax=ax, alpha=0.8)
ax.set_title('Impact of Publication Delays on Q4 2023 Data Availability\n(as of December 1, 2023)')
ax.set_ylabel('Data Availability (%)')
ax.set_xlabel('Economic Indicators')
ax.legend(title='Delay Scenarios')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)

# Add horizontal line at 100%
ax.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Complete availability')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Summary and Key Insights

This notebook demonstrated the key functionality of the `tsforecast.delays` module:

In [ ]:
print("=== SUMMARY OF DELAYS MODULE DEMONSTRATION ===")
print()
print("1. DATA MANAGEMENT:")
print(f"   - Created synthetic monthly and quarterly economic data")
print(f"   - Simulated data release scenarios with publication delays")
print(f"   - Identified {monthly_comparison['new_observations_count'] + quarterly_comparison['new_observations_count']} new observations")
print()
print("2. DELAY CALCULATION:")
print(f"   - Calculated delays for {len(median_delays)} different indicators")
print(f"   - Median delays range from {min(median_delays.values()):.1f} to {max(median_delays.values()):.1f} days")
print()
print("3. TRANSFORMATION MODES:")
print(f"   - 'shift': Moves data forward in time by delay amount")
print(f"   - 'mask': Hides data that wouldn't be available yet")
print()
print("4. PRACTICAL IMPACT:")
print(f"   - In realistic delay scenario, only {availability_df.loc[:, 'Realistic delays'].mean():.1f}% of Q4 data available")
print(f"   - High-frequency indicators (unemployment) more timely than low-frequency (GDP)")
print(f"   - Delays significantly impact nowcasting capabilities")
print()
print("5. MODULE FEATURES:")
print(f"   - ✓ Sklearn-compatible transformers")
print(f"   - ✓ Support for mixed frequency data")
print(f"   - ✓ Panel data support (multi-country, multi-indicator)")
print(f"   - ✓ Reversible transformations")
print(f"   - ✓ Comprehensive delay statistics")

## Next Steps

This module can be extended for:

1. **Real-time nowcasting pipelines** with automatic delay adjustment
2. **Multi-country analysis** with country-specific delay patterns  
3. **Machine learning pipelines** where delays are automatically applied during training
4. **Forecast evaluation** accounting for realistic data availability constraints
5. **Policy analysis** measuring the value of more timely data releases

The delays module provides a robust foundation for handling the practical constraints of real-time economic analysis.